In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import pickle
import sys
sys.path.append("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/")
from benchmarker import Benchmarker

/mnt/datadisk/lizhongzhan/miniconda3/envs/benchmark_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/datadisk/lizhongzhan/miniconda3/envs/benchmark_env/lib/python3.10/site-packages/umap/__init__.py:9: ImportWarning: Tensorflow not installed; ParametricUMAP will be unavailable
  warn(


In [3]:
import os
os.chdir("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Mouse_brain_unpaired2")

Prepare data

In [4]:
rna = sc.read_h5ad("rna.h5ad")
atac = sc.read_h5ad("atac.h5ad")
gs = sc.read_h5ad("gs.h5ad")
gs.var_names = [i.lower().capitalize() for i in gs.var_names ]

In [5]:
gs = gs[rna.obs_names]
gs.obsm["spatial"] = rna.obsm["spatial"].copy()

In [6]:
# import h5py
# data = h5py.File("annot.h5", "r")
# cell = np.array(data["Cell"]).astype(str)
# annot = np.array(data["LayerName"]).astype(str)

In [7]:
# df = pd.DataFrame(
#     {
#         "cell":  cell,
#         "annot": annot
#     },
#     index=cell
# )
# df.index = [i.split("-")[0] for i in df.index]
# df = df.reindex(rna.obs_names)
# rna.obs["cell_type"] = df["annot"].copy()

In [8]:
gs.obsm["spatial"] = rna.obsm["spatial"].copy()

In [9]:
# comm_gene = pd.Index(set(rna.var_names) & set(gs.var_names))
# rna = rna[:,comm_gene]
# gs = gs[:,comm_gene]

In [10]:
(gs.obs_names == atac.obs_names).all()

np.True_

In [11]:
import anndata as ad
import h5py
import numpy as np
from scipy import sparse

def h5ad_to_h5(adata, output_file: str):

    if adata.raw is not None and adata.raw.X is not None:
        X = adata.raw.X
        features = np.asarray(adata.raw.var_names, dtype=str)
    else:
        X = adata.X
        features = np.asarray(adata.var_names, dtype=str)

    barcodes = np.asarray(adata.obs_names, dtype=str)

    spatial = None
    if "spatial" in adata.obsm:
        spatial = np.asarray(adata.obsm["spatial"], dtype=np.float32)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = sparse.csr_matrix(X)

    if X.data.size == 0 or (np.all(X.data >= 0) and np.all(np.isclose(X.data, np.round(X.data)))):
        X.data = X.data.astype(np.int32, copy=False)
    else:
        X.data = X.data.astype(np.float32, copy=False)

    with h5py.File(output_file, "w") as f:
        g = f.create_group("matrix")

        g.create_dataset("data", data=X.data,
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indices", data=X.indices.astype(np.int32, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indptr", data=X.indptr.astype(np.int64, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("shape", data=np.asarray(X.shape, dtype=np.int64))

        g.create_dataset("barcodes", data=np.array(barcodes, dtype="S"))
        g.create_dataset("features", data=np.array(features, dtype="S"))

        if spatial is not None:
            g.create_dataset(
                "spatial",
                data=spatial,
                compression="gzip",
                compression_opts=4,
                shuffle=True
            )

In [12]:
# h5ad_to_h5(rna, output_file="rna.h5")
# h5ad_to_h5(atac, output_file="atac.h5")
# h5ad_to_h5(gs, output_file="gs.h5")

In [13]:
bm = Benchmarker(R_conda_env="Rbase")

Run evaluation methods

In [14]:
# data_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Mouse_brain_unpaired2/"
# bm.run(methods=["scConfluence"],
#        RNA_file_path=data_folder+"rna.h5",
#        ATAC_file_path=data_folder+"/atac.h5",
#        ADT_file_path=data_folder+"/gs.h5", # path for gene score
#        n_cluster=18,
#        conda_envs={"Monae": "scSLAT", "SIMBA": "scSLAT", "SCALEX": "scSLAT", "GLUE": "scSLAT",
#        "MaxFuse": "scSLAT", "LIGER": "scSLAT", "scConfluence": "cell2loc_env",},
#        save_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsUnpaired/Mouse_brain2")

In [15]:
# res = pd.read_csv("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsUnpaired/Mouse_brain/glue.csv", index_col=0)
# combined = sc.concat([rna, atac], label="omics")
# combined.obsm["spatial"][:,0] = combined.obsm["spatial"][:,0]*-1
# combined.obsm["spatial"][:,1] = combined.obsm["spatial"][:,1]*-1

In [16]:
# res.columns = ["UMAP1", "UMAP2", "cluster"]
# combined.obsm["X_umap"] = np.array(res[["UMAP1","UMAP2"]])
# combined.obs["cluster"] = [str(i) for i in list(res["cluster"])]

In [17]:
sc.set_figure_params(dpi=100, figsize=(4,4), facecolor="white")

In [18]:
# sc.pl.umap(combined, color=["cluster", "omics"])
# t_rna = combined[combined.obs["omics"]=="0",]
# t_atac = combined[combined.obs["omics"]=="1",]
# sc.pl.embedding(t_rna, color="cluster", size=100, basis="spatial")
# sc.pl.embedding(t_atac, color="cluster", size=100, basis="spatial")

Evaluation

In [19]:
import h5py
data = h5py.File("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Mouse_brain_unpaired2/annot.h5", "r")
cell = np.array(data["Cell"]).astype(str)
annot = np.array(data["LayerName"]).astype(str)
df = pd.DataFrame(
    {
        "cell":  cell,
        "annot": annot
    },
    index=cell
)
df.index = [i.split("-")[0] for i in df.index]
df = df.reindex(rna.obs_names)
rna.obs["cell_type"] = df["annot"].copy()

In [20]:
gs = gs[rna.obs_names]
gs.obs["cell_type"] = rna.obs["cell_type"].copy()

In [21]:
result_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsUnpaired/Mouse_brain2/"
methods = ["GLUE", "Monae", "SIMBA", "SCALEX", "MaxFuse", "LIGER", "scConfluence", "Seurat_CCA",
"Seurat_RPCA", "BindSC", "switch"]
res = bm.read_result(path=result_folder,
                     methods=methods,
                     reindex=False)

In [22]:
combined = sc.concat([rna, gs], label="omics")

In [23]:
# metrics = bm.cal_metrics(adata=combined, batch_key="omics", label_key="cell_type",
#                          res_dict=res, methods=methods, verbose=True, rep=1,
#                          min_max_scale=False,
#                          save=f"{result_folder}/metrics.pkl")

In [24]:
with open(f"{result_folder}/metrics.pkl", "rb") as f:
    metrics = pickle.load(f)
metric = metrics[0]

In [25]:
bm.set_plot_params(params_dict={"figure.dpi": 300},
# font_file_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Helvetica.ttf"
)

In [26]:
figure_save_dir = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/figures/Multi_omics_unpaired/Mouse_brain2"

In [27]:
metric = metrics[0]
metric.index = metric.index.map({
    i: i if i != "switch" else "SWITCH" for i in metric.index
})

In [28]:
# bm.plot_heatmap(metric_df=metric, total_name="Total",
#                 save=f"{figure_save_dir}/summary_heatmap.pdf",
#                 # show_top=7,
#                 # show_bottom=0,
#                 # insert_marker_row = 8,
#                 )

In [29]:
from benchmarker import split_adata, transform_coord
import numpy as np
spatial = [i.obsm["spatial"] for i in split_adata(combined, batch_key="omics")]
spatial = transform_coord(spatial, vertical=True, axis="y", horizontal=False, angle=0)

In [30]:
spatial_methods = ["switch"]
bg_dict = {i:"#D4B483" if i in spatial_methods else "#5873a4" for i in bm.all_methods }
bg_dict["RNA"] = "#97a4af"
bg_dict["ATAC"] = "#97a4af"
bg_dict["Annotation"] = "#97a4af"
bg_dict["SWITCH"] = "#D4B483"
bg_dict["Modality"] = "#97a4af"
bg_dict["Cell type"] = "#97a4af"

In [31]:
from benchmarker import get_scatter_cmap
palette = get_scatter_cmap(sorted([str(i) for i in list(range(18))], reverse=False))

In [32]:
# for m in methods:
#     res = pd.read_csv(f"/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsUnpaired/Mouse_brain2/{m.lower()}.csv", index_col=0)
#     res.columns = ["UMAP1", "UMAP2", "cluster"]
#     cluster = len(set(res["cluster"]))
#     print(m, cluster)

In [33]:
# res_dir = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsUnpaired/Mouse_brain2/"
# def search_resolution(adata, fixed_clus_count, increment=0.01):
#     closest_count = np.inf  
#     closest_res = None  
    
#     for res in sorted(list(np.arange(0.1, 2.5, increment)), reverse=True):
#         sc.tl.leiden(adata, random_state=0, resolution=res, key_added="temp_label")
#         count_unique_leiden = len(list(set(adata.obs["temp_label"])))
#         current_diff = abs(count_unique_leiden - fixed_clus_count)
#         if current_diff < closest_count:
#             closest_count = current_diff
#             closest_res = res
#         if count_unique_leiden == fixed_clus_count:
#             break

#     return closest_res
# def recluster(methods, ncluster):
#     df = pd.read_csv(f"{res_dir}/{methods[0].lower()}_latent.csv", index_col=0)
#     adata = sc.AnnData(X=np.zeros((df.shape[0], 10)))
#     for m in methods:
#         adata.obsm[f"X_{m}"] = np.array(pd.read_csv(f"{res_dir}/{m.lower()}_latent.csv", index_col=0))
#         sc.pp.neighbors(adata, use_rep=f"X_{m}")
#         sc.tl.umap(adata)
#         res = search_resolution(adata, fixed_clus_count=ncluster)
#         sc.tl.leiden(adata, resolution=res, key_added="cluster")
#         print(m, len(set(adata.obs["cluster"])))
#         umap = pd.DataFrame(adata.obsm["X_umap"], columns=["UMAP1", "UMAP2"], index=adata.obs_names)
#         umap.insert(2, "cluster", adata.obs['cluster'].values)
#         umap.to_csv(os.path.join(res_dir, m.lower() + ".csv"))

In [34]:
# recluster(["BindSC"], ncluster=18)

In [35]:
palette_annot = {
    "ACB": "#bcbd22",
    "CP": "#e377c2",
    "L1-L3": "#ff7f0e",
    "L4": "#2ca02c",
    "L5": "#d62728",
    "L6a/b": "#9467bd",
    "VL": "#7f7f7f",
    "ccg/aco": "#8c564b",
    "others": "#1f77b4"
}

In [36]:
palette_cluster = {}
for i, annot in enumerate(sorted(set(rna.obs["cell_type"]))):
    if annot == "1_others":
        annot = "others"
    palette_cluster[str(i)] = palette_annot[annot] 

In [37]:
# bm.plot_spatial(spatial=[spatial[0]],
#                 label_dict={"annot": np.array(rna.obs["cell_type"]).reshape(-1,1)},
#                 figsize=(1.97, 2.03),
#                 frameon=True,
#                 inner_gs_row=1, inner_gs_col=1,
#                 size=7.5,
#                 ncol=1,
#                 xlabel=["Annotation"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 outer_row_hspace=0.15,
#                 outer_col_wspace=0.1,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.015,
#                 save_dpi=600,
#                 palette=palette_cluster,
#                 save=f"{figure_save_dir}/spatial_annot.pdf"
#                 )

In [41]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(14, 4.2),
#                 frameon=True,
#                 inner_gs_row=2,
#                 inner_gs_col=1,
#                 size=6,
#                 ncol=7,
#                 xlabel=["SWITCH", "GLUE", "Monae", "MaxFuse", "BindSC", "SCALEX", "scConfluence"],
#                 ylabel=["RNA", "ATAC"],
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["switch", "GLUE", "Monae", "MaxFuse", "BindSC", "SCALEX", "scConfluence"],
#                 outer_row_hspace=0,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 palette=palette,
#                 # inner_col_wspace = -0.12,
#                 ylabel_pad = 0.018,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.013, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_methods.pdf",
#                 rasterized=True,
#                 )

In [43]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(14, 8.6),
#                 frameon=True,
#                 inner_gs_row=2,
#                 inner_gs_col=1,
#                 size=6,
#                 ncol=7,
#                 xlabel=["SWITCH", "GLUE", "Monae", "MaxFuse", "BindSC", "SCALEX", "scConfluence", "Seurat_RPCA",  "Seurat_CCA","SIMBA",
#                 "LIGER"],
#                 ylabel=["RNA", "ATAC"],
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["switch", "GLUE", "Monae", "MaxFuse", "BindSC", "SCALEX", "scConfluence", "Seurat_RPCA",  "Seurat_CCA","SIMBA",
#                 "LIGER"],
#                 outer_row_hspace=0.12,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 palette=palette,
#                 # inner_col_wspace = -0.12,
#                 ylabel_pad = 0.0168,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.0125, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_methods_all.pdf",
#                 rasterized=True,
#                 )

In [127]:
res["Batch"] = {}
for m in methods:
    res["Batch"][m] = np.array(combined.obs["omics"]).astype(str)

In [126]:
combined.obs["cell_type"] = combined.obs["cell_type"].map({
    i: i if i !="1_others" else "others" for i in set(combined.obs["cell_type"])
})

In [60]:
palette_annot = {}
for i in range(9):
    palette_annot[sorted(set(combined.obs["cell_type"]))[i]] = get_scatter_cmap([str(i) for i in list(range(9))])[str(i)]

In [129]:
# bm.plot_umap(embed_dict=res["UMAP"],
#              batch_dict=res["Batch"],
#              annot_list=list(combined.obs["cell_type"]),
#              figsize=(14, 8.2),
#              frameon=True,
#              inner_gs_row=2,
#              inner_gs_col=1,
#              size=5.5,
#              ncol=7,
#              xlabel=["SWITCH", "GLUE", "Monae", "MaxFuse", "BindSC", "SCALEX", "scConfluence", "Seurat_RPCA",  "Seurat_CCA","SIMBA",
#                 "LIGER"],
#              only_show_top=False,
#              ylabel=["Modality", "Cell type"],
#              only_show_left=True,
#              background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#              order=["switch", "GLUE", "Monae", "MaxFuse", "BindSC", "SCALEX", "scConfluence", "Seurat_RPCA",  "Seurat_CCA","SIMBA",
#                 "LIGER"],
#              axis_width=1.2,
#              axis_color="lightgrey",
#              outer_col_wspace=0.05,
#              save_dpi=600,
#              ylabel_pad=0.0168,
#              xlabel_pad=0.013,
#              outer_row_hspace=0.12,
#              merge=True,
#              merge_margin_size=0.4,
#              palettes=[None, palette_annot],
#              save=f"{figure_save_dir}/umap_methods_all.pdf"
# )

In [131]:
# bm.plot_umap(embed_dict=res["UMAP"],
#              batch_dict=res["Batch"],
#              annot_list=list(combined.obs["cell_type"]),
#              figsize=(14, 4.2),
#              frameon=True,
#              inner_gs_row=2,
#              inner_gs_col=1,
#              size=5.5,
#              ncol=7,
#              xlabel=["SWITCH", "GLUE", "Monae", "MaxFuse", "BindSC", "SCALEX", "scConfluence"],
#              only_show_top=False,
#              ylabel=["Modality", "Cell type"],
#              only_show_left=True,
#              background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#              order=["switch", "GLUE", "Monae", "MaxFuse", "BindSC", "SCALEX", "scConfluence"],
#              axis_width=1.2,
#              axis_color="lightgrey",
#              outer_col_wspace=0.05,
#              save_dpi=600,
#              ylabel_pad=0.0168,
#              xlabel_pad=0.013,
#              outer_row_hspace=0.22,
#              merge=True,
#              merge_margin_size=0.4,
#              palettes=[None, palette_annot],
#              save=f"{figure_save_dir}/umap_methods.pdf"
# )

In [207]:
# palette_annot2 = {}
# for key in palette_annot.keys():
#     if key[0]=="L":
#         palette_annot2[key] =  palette_annot[key]
#     else:
#         palette_annot2[key.lower()] = palette_annot[key]
# palette_annot2

In [205]:
# bm.plot_legend(color_map=palette_annot2, marker="o", ncol=1,
# save=f"{figure_save_dir}/annot_legend.pdf",)

In [206]:
# bm.plot_legend(category_lst=rna.obs["cell_type"].map({
#     "ACB": "acb",
#     "CP": "cp",
#     "L1-L3": "L1-L3",
#     "L4": "L4",
#     "L5": "L5",
#     "L6a/b": "L6a/b",
#     "VL": "vl",
#     "ccg/aco": "ccg/aco",
#     "1_others": "others"
# }), marker="o", ncol=1,
# save=f"{figure_save_dir}/annot_legend.pdf",)

In [184]:
# bm.plot_legend(category_lst=[str(i) for i in range(18)],
#                 marker="o",
#                 ncol=6,
#                 save=f"{figure_save_dir}/cluster_legend.pdf",
#                 order=sorted([str(i) for i in range(18)], key=lambda x: int(x))
#                 )